In [1]:
import pandas as pd

In [3]:
df= pd.read_excel('nadil_category_expenses.xlsx')

In [4]:
print(df.head())

        Date                Discription  Payments  Receipts   Balance  \
0 2022-11-06           t ahirt OTHBNK T    6030.0       NaN  12454.64   
1 2022-11-06   010001088282101 OTHBNK T    3030.0       NaN   9424.64   
2 2022-11-15  RIB/RMB SE.CH 20 IBMB Chg      25.0       NaN   9399.64   
3 2022-11-18  nadil Siriwardha MB SA TF     450.0       NaN  11537.14   
4 2022-12-24             nadil OTHBNK T    7530.0       NaN   27264.9   

            cleaned_particulars                  Category  Cluster  
0              t ahirt othbnk t            NADIL OTHBNK T        0  
1      010001088282101 othbnk t                  OTHBNK T        1  
2  rib/rmb se.ch 20 ibmb charge  RIBRMB SECH  IBMB CHARGE        2  
3     nadil siriwardha mb sa tf            NADIL OTHBNK T        0  
4                nadil othbnk t            NADIL OTHBNK T        0  


In [5]:
print(df.dtypes)

Date                   datetime64[ns]
Discription                    object
Payments                      float64
Receipts                      float64
Balance                        object
cleaned_particulars            object
Category                       object
Cluster                         int64
dtype: object


In [6]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 521 entries, 0 to 520
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 521 non-null    datetime64[ns]
 1   Discription          521 non-null    object        
 2   Payments             521 non-null    float64       
 3   Receipts             0 non-null      float64       
 4   Balance              521 non-null    object        
 5   cleaned_particulars  521 non-null    object        
 6   Category             521 non-null    object        
 7   Cluster              521 non-null    int64         
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.7+ KB
None


In [7]:
# Remove commas or other non-numeric stuff if needed, then convert to float
df['Balance'] = pd.to_numeric(df['Balance'].astype(str).str.replace(',', ''), errors='coerce')


In [8]:
print(df['Balance'].dtypes)


float64


In [14]:
# First, make sure Date is just date (not full datetime)
df['Date'] = pd.to_datetime(df['Date']).dt.date

# Group by Date and Clusater
aggregated = df.groupby(['Date', 'Cluster']).agg({
    'Payments': 'sum',
    'Receipts': 'sum',
    'Balance': 'last',  # last balance for that day+cluster
    'Category': 'first',  # optional: pick the first category seen
    'cleaned_particulars': 'first'  # optional
}).reset_index()


ValueError: You are trying to merge on datetime64[ns] and object columns for key 'Date'. If you wish to proceed you should use pd.concat

In [13]:
print(aggregated.head())
print(aggregated.dtypes)

         Date  Cluster  Payments  Receipts   Balance  \
0  2022-11-06        0    6030.0       0.0  12454.64   
1  2022-11-06        1    3030.0       0.0   9424.64   
2  2022-11-15        2      25.0       0.0   9399.64   
3  2022-11-18        0     450.0       0.0  11537.14   
4  2022-12-24        0    7530.0       0.0  27264.90   

                   Category           cleaned_particulars  
0            NADIL OTHBNK T              t ahirt othbnk t  
1                  OTHBNK T      010001088282101 othbnk t  
2  RIBRMB SECH  IBMB CHARGE  rib/rmb se.ch 20 ibmb charge  
3            NADIL OTHBNK T     nadil siriwardha mb sa tf  
4            NADIL OTHBNK T                nadil othbnk t  
Date                    object
Cluster                  int64
Payments               float64
Receipts               float64
Balance                float64
Category                object
cleaned_particulars     object
dtype: object


In [15]:
all_dates = pd.date_range(start=aggregated['Date'].min(), end=aggregated['Date'].max())

In [16]:
all_clusters = aggregated['Cluster'].unique()

In [17]:
full_index = pd.MultiIndex.from_product([all_dates, all_clusters], names=['Date', 'Cluster'])

In [18]:
full_df = pd.DataFrame(index=full_index).reset_index()

In [25]:
#convert Dates into datetime forments
full_df['Date'] = pd.to_datetime(full_df['Date'])


In [26]:
#convert aggreagated date into datetime format
aggregated['Date'] = pd.to_datetime(aggregated['Date'])

In [27]:
full_df = full_df.merge(aggregated[['Date', 'Cluster', 'Payments', 'Receipts', 'Category', 'cleaned_particulars']], 
                        on=['Date', 'Cluster'], how='left')


In [28]:
# Step 3: Fill missing values
full_df['Payments'] = full_df['Payments'].fillna(0)
full_df['Receipts'] = full_df['Receipts'].fillna(0)
full_df['Category'] = full_df['Category'].fillna('Unknown')
full_df['cleaned_particulars'] = full_df['cleaned_particulars'].fillna('')

In [29]:
# Step 4: Calculate Net = Receipts - Payments
full_df = full_df.sort_values(by='Date')
full_df['Net'] = full_df['Receipts'] - full_df['Payments']

In [30]:
# Step 5: Compute global Balance
# Get starting balance from your original data (earliest one)
starting_balance = df.sort_values('Date').iloc[0]['Balance']

In [31]:
# Cumulative net change
daily_net = full_df.groupby('Date')['Net'].sum().reset_index()
daily_net['Balance'] = starting_balance + daily_net['Net'].cumsum()

In [32]:
# Merge back to full_df by Date
full_df = full_df.merge(daily_net[['Date', 'Balance']], on='Date', how='left')

In [33]:
print(full_df.head())

        Date  Cluster  Payments_x  Receipts_x      Category_x  \
0 2022-11-06        0      6030.0         0.0  NADIL OTHBNK T   
1 2022-11-06       11         NaN         NaN             NaN   
2 2022-11-06       10         NaN         NaN             NaN   
3 2022-11-06       -1         NaN         NaN             NaN   
4 2022-11-06        8         NaN         NaN             NaN   

  cleaned_particulars_x  Payments_y  Receipts_y      Category_y  \
0      t ahirt othbnk t      6030.0         0.0  NADIL OTHBNK T   
1                   NaN         NaN         NaN             NaN   
2                   NaN         NaN         NaN             NaN   
3                   NaN         NaN         NaN             NaN   
4                   NaN         NaN         NaN             NaN   

  cleaned_particulars_y  Payments  Receipts        Category  \
0      t ahirt othbnk t    6030.0       0.0  NADIL OTHBNK T   
1                   NaN       0.0       0.0         Unknown   
2                

In [34]:
print(full_df.dtypes)

Date                     datetime64[ns]
Cluster                           int64
Payments_x                      float64
Receipts_x                      float64
Category_x                       object
cleaned_particulars_x            object
Payments_y                      float64
Receipts_y                      float64
Category_y                       object
cleaned_particulars_y            object
Payments                        float64
Receipts                        float64
Category                         object
cleaned_particulars              object
Net                             float64
Balance                         float64
dtype: object


In [35]:
final_df = full_df[['Date', 'Cluster', 'Payments', 'Receipts', 'Balance']].copy()


In [39]:
print(final_df.head(100))


         Date  Cluster  Payments  Receipts  Balance
0  2022-11-06        0    6030.0       0.0  3394.64
1  2022-11-06       11       0.0       0.0  3394.64
2  2022-11-06       10       0.0       0.0  3394.64
3  2022-11-06       -1       0.0       0.0  3394.64
4  2022-11-06        8       0.0       0.0  3394.64
..        ...      ...       ...       ...      ...
95 2022-11-13        7       0.0       0.0  3394.64
96 2022-11-13        6       0.0       0.0  3394.64
97 2022-11-13       11       0.0       0.0  3394.64
98 2022-11-13        4       0.0       0.0  3394.64
99 2022-11-13        3       0.0       0.0  3394.64

[100 rows x 5 columns]


In [40]:
import pandas as pd
import numpy as np

# Step 0: Ensure correct types
df['Date'] = pd.to_datetime(df['Date'])
df['Payments'] = df['Payments'].fillna(0)
df['Receipts'] = df['Receipts'].fillna(0)

# Step 1: Aggregate df to sum per Date + Cluster
aggregated = df.groupby(['Date', 'Cluster']).agg({
    'Payments': 'sum',
    'Receipts': 'sum',
    'Category': 'first',
    'cleaned_particulars': 'first'
}).reset_index()

# Step 2: Create full grid of all Date × Cluster combinations
all_dates = pd.date_range(start=df['Date'].min(), end=df['Date'].max())
all_clusters = df['Cluster'].unique()
full_index = pd.MultiIndex.from_product([all_dates, all_clusters], names=['Date', 'Cluster'])
full_df = pd.DataFrame(index=full_index).reset_index()

# Step 3: Merge full grid with aggregated data
full_df = full_df.merge(aggregated, on=['Date', 'Cluster'], how='left')

# Step 4: Fill missing Payments/Receipts with 0, and others as needed
full_df['Payments'] = full_df['Payments'].fillna(0)
full_df['Receipts'] = full_df['Receipts'].fillna(0)
full_df['Category'] = full_df['Category'].fillna('Unknown')
full_df['cleaned_particulars'] = full_df['cleaned_particulars'].fillna('')

# Step 5: Sort by Date then Cluster (for stability)
full_df = full_df.sort_values(['Date', 'Cluster']).reset_index(drop=True)

# Step 6: Compute Net Flow
full_df['Net'] = full_df['Receipts'] - full_df['Payments']

# Step 7: Compute Global Running Balance
# Get starting balance from the *first row of original df sorted by date*
starting_balance = df.sort_values('Date').iloc[0]['Balance']
full_df['Balance'] = starting_balance + full_df['Net'].cumsum()

# Step 8: Final Clean DataFrame
final_df = full_df[['Date', 'Cluster', 'Payments', 'Receipts', 'Balance']].copy()


In [41]:
print(final_df.head(100))

         Date  Cluster  Payments  Receipts   Balance
0  2022-11-06       -1       0.0       0.0  12454.64
1  2022-11-06        0    6030.0       0.0   6424.64
2  2022-11-06        1    3030.0       0.0   3394.64
3  2022-11-06        2       0.0       0.0   3394.64
4  2022-11-06        3       0.0       0.0   3394.64
..        ...      ...       ...       ...       ...
95 2022-11-13        3       0.0       0.0   3394.64
96 2022-11-13        4       0.0       0.0   3394.64
97 2022-11-13        5       0.0       0.0   3394.64
98 2022-11-13        6       0.0       0.0   3394.64
99 2022-11-13        7       0.0       0.0   3394.64

[100 rows x 5 columns]


In [42]:
#original dataframe describe
print(df.describe())

                                Date      Payments  Receipts       Balance  \
count                            521    521.000000     521.0    520.000000   
mean   2024-03-01 05:42:43.531669760   1811.736641       0.0  11728.292846   
min              2022-11-06 00:00:00      0.230000       0.0    633.100000   
25%              2023-10-09 00:00:00    150.000000       0.0   1389.225000   
50%              2024-03-02 00:00:00   1030.000000       0.0   3693.430000   
75%              2024-07-31 00:00:00   2225.000000       0.0  11499.197500   
max              2025-01-31 00:00:00  21200.000000       0.0  75138.990000   
std                              NaN   2516.379879       0.0  17791.798717   

          Cluster  
count  521.000000  
mean     6.523992  
min     -1.000000  
25%      6.000000  
50%      6.000000  
75%      9.000000  
max     11.000000  
std      2.345187  


In [43]:
#final dataframe describe
print(final_df.describe())

                      Date       Cluster      Payments  Receipts  \
count                10634  10634.000000  10634.000000   10634.0   
mean   2023-12-19 12:00:00      5.000000     88.763851       0.0   
min    2022-11-06 00:00:00     -1.000000      0.000000       0.0   
25%    2023-05-29 00:00:00      2.000000      0.000000       0.0   
50%    2023-12-19 12:00:00      5.000000      0.000000       0.0   
75%    2024-07-11 00:00:00      8.000000      0.000000       0.0   
max    2025-01-31 00:00:00     11.000000  21200.000000       0.0   
std                    NaN      3.741833    720.579652       0.0   

             Balance  
count   10634.000000  
mean  -435876.990429  
min   -931460.150000  
25%   -687180.530000  
50%   -442049.060000  
75%   -144455.320000  
max     12454.640000  
std    290554.890654  


In [44]:
# Step 1: Compute Net per row
final_df['Net'] = final_df['Receipts'] - final_df['Payments']

# Step 2: Sum Net per date (once per day)
daily_net = final_df.groupby('Date')['Net'].sum().reset_index()

# Step 3: Get starting balance
starting_balance = df.sort_values('Date').iloc[0]['Balance']

# Step 4: Compute cumulative balance per day
daily_net['Balance'] = starting_balance + daily_net['Net'].cumsum()

# Step 5: Merge this daily balance back to final_df
final_df = final_df.drop(columns='Balance')  # drop old broken balance
final_df = final_df.merge(daily_net[['Date', 'Balance']], on='Date', how='left')


In [45]:
# Final DataFrame with correct balance
print(final_df.head(100))

         Date  Cluster  Payments  Receipts     Net  Balance
0  2022-11-06       -1       0.0       0.0     0.0  3394.64
1  2022-11-06        0    6030.0       0.0 -6030.0  3394.64
2  2022-11-06        1    3030.0       0.0 -3030.0  3394.64
3  2022-11-06        2       0.0       0.0     0.0  3394.64
4  2022-11-06        3       0.0       0.0     0.0  3394.64
..        ...      ...       ...       ...     ...      ...
95 2022-11-13        3       0.0       0.0     0.0  3394.64
96 2022-11-13        4       0.0       0.0     0.0  3394.64
97 2022-11-13        5       0.0       0.0     0.0  3394.64
98 2022-11-13        6       0.0       0.0     0.0  3394.64
99 2022-11-13        7       0.0       0.0     0.0  3394.64

[100 rows x 6 columns]
